# SPLADE-max: обучение + BEIR-eval каждые 1000 шагов

Клон `splade_experiments.ipynb` с одним экспериментом (**SPLADE-max**), гиперпараметрами
статьи (100k шагов, batch 124) и zero-shot проверкой на **NFCorpus + SciFact** каждые
1000 шагов. Модели на диск **не сохраняются** — eval идёт in-memory.

Логи: `outputs/logs/<run_id>/step_XXXXXX.json` (100 файлов, фиксированный формат).

Порядок:
1. Setup. 2. Конфиг данных. 3. Конфиг эксперимента. 4. MS MARCO. 5. BEIR.
6. Eval-хелперы. 7. Обучение. 8. Сводка по логам.


## 1. Setup


In [2]:
# Первый запуск на новой машине (в активированном venv) — раскомментировать:
# %pip install -r requirements.txt
%load_ext autoreload
%autoreload 2

import gc
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer

from splade_lab import benchmark, data, datasets, evaluate
from splade_lab.config import OUTPUTS_DIR, merge_config
from splade_lab.model import Splade, flops_loss, tok
from splade_lab.progress import tqdm
from splade_lab.train import pick_device, set_seed

device = pick_device()
print(f"torch {torch.__version__} | устройство: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))


torch 2.5.1+cu124 | устройство: cpu


## 2. Конфиг данных

`full`: triples ≈ max_steps × batch_size (12.4M). Обучение циклически перемешивает
triples — если строк меньше, данные повторяются.


In [3]:
MODE = "full"  # "smoke" | "full"

DATA = {
    "urls": {
        "triples": "https://msmarco.z22.web.core.windows.net/msmarcoranking/triples.train.small.tar.gz",
        "collection": "https://msmarco.z22.web.core.windows.net/msmarcoranking/collection.tar.gz",
        "queries": "https://msmarco.z22.web.core.windows.net/msmarcoranking/queries.tar.gz",
        "qrels_dev": "https://msmarco.z22.web.core.windows.net/msmarcoranking/qrels.dev.small.tsv",
    },
    "data_dir": "data/msmarco",
    "smoke": {
        "num_train_triples": 256,
        "num_eval_queries": 20,
        "num_corpus_docs": 1000,
    },
    "full": {
        "num_train_triples": 12_400_000,
        "num_eval_queries": -1,
    },
}

print(f"MODE = {MODE} | параметры данных: {DATA[MODE]}")


MODE = full | параметры данных: {'num_train_triples': 12400000, 'num_eval_queries': -1}


## 3. Конфиг эксперимента (только SPLADE-max)

Гиперпараметры SPLADE v2 / статьи: distilbert, Adam 2e-5, warmup 6000, batch 124,
100k шагов, FLOPS warmup 10k.


In [4]:
BASE = {
    "model": {
        "hf_model": "distilbert-base-uncased",
        "query_encoder": "mlm",
        "max_len_query": 32,
        "max_len_doc": 256,
    },
    "train": {
        "seed": 4,
        "lr": 2e-5,
        "batch_size": 32,
        "max_steps": 400_000,
        "warmup_steps": 12_000,
        "flops_warmup_steps": 20_000,
        "lambda_q": 3e-4,
        "lambda_d": 1e-4,
        "log_every": 100,
    },
    "eval": {
        "batch_size_docs": 256,
        "batch_size_queries": 32,
        "batch_size_search": 32,
        "recall_ks": [10, 100],
    },
}

SMOKE_OVERRIDES = {
    "model": {"max_len_doc": 128},
    "train": {"max_steps": 20, "batch_size": 8, "warmup_steps": 4,
              "flops_warmup_steps": 8, "log_every": 5},
    "eval": {"batch_size_docs": 32, "batch_size_queries": 32, "recall_ks": [10, 100]},
}

EXPERIMENT = "v1_splade_max"
EVAL_EVERY = 1000
EXAMPLE_QIDS = 10
RANK_TOPK = 10
RECALL_KS = (10, 100)
NDCG_KS = (10,)

def build_config() -> dict:
    cfg = merge_config(BASE, {})
    if MODE == "smoke":
        cfg = merge_config(cfg, SMOKE_OVERRIDES)
        global EVAL_EVERY
        EVAL_EVERY = 10
    cfg["version"], cfg["mode"] = EXPERIMENT, MODE
    return cfg

cfg = build_config()
print(json.dumps(cfg, indent=2, ensure_ascii=False))
print(f"\nEVAL_EVERY = {EVAL_EVERY} | BEIR-логов ожидается: {cfg['train']['max_steps'] // EVAL_EVERY}")


{
  "model": {
    "hf_model": "distilbert-base-uncased",
    "query_encoder": "mlm",
    "max_len_query": 32,
    "max_len_doc": 256
  },
  "train": {
    "seed": 4,
    "lr": 2e-05,
    "batch_size": 32,
    "max_steps": 400000,
    "warmup_steps": 12000,
    "flops_warmup_steps": 20000,
    "lambda_q": 0.0003,
    "lambda_d": 0.0001,
    "log_every": 100
  },
  "eval": {
    "batch_size_docs": 256,
    "batch_size_queries": 32,
    "batch_size_search": 32,
    "recall_ks": [
      10,
      100
    ]
  },
  "version": "v1_splade_max",
  "mode": "full"
}

EVAL_EVERY = 1000 | BEIR-логов ожидается: 400


## 4. Данные MS MARCO (скачиваются один раз)


In [4]:
from splade_lab import data as msdata

prepare = msdata.prepare_full if MODE == "full" else msdata.prepare_smoke
msmarco_dir = prepare(DATA)

print(f"\nКаталог: {msmarco_dir}")
print("Строк в файлах:", msdata.dataset_stats(msmarco_dir))


[data] full уже готов: /home/uvuv/workspace/sandbox_03/data/msmarco/full

Каталог: /home/uvuv/workspace/sandbox_03/data/msmarco/full


Строк в файлах: {'collection.tsv': 8841823, 'queries.tsv': 6980, 'qrels.tsv': 7437, 'triples.tsv': 3200000}


## 5. BEIR: NFCorpus + SciFact

Наборы подготавливаются один раз; тексты и qrels кешируются в памяти для быстрого
eval без повторного чтения TSV.


In [8]:
BEIR_NAMES = ("nfcorpus", "scifact")
BEIR_CFG = {n: datasets.beir_config(n, num_eval_queries=-1) for n in BEIR_NAMES}
BEIR_DIRS = {n: datasets.prepare_beir(BEIR_CFG[n]) for n in BEIR_NAMES}

BEIR_CACHE = {}
for name in BEIR_NAMES:
    ds_dir = BEIR_DIRS[name]
    pids, doc_texts = data.load_collection(ds_dir)
    qids, q_texts = data.load_queries(ds_dir)
    qrels = benchmark.load_graded_qrels(ds_dir)
    example_qids = [q for q in qids if q in qrels][:EXAMPLE_QIDS]
    BEIR_CACHE[name] = {
        "ds_dir": ds_dir,
        "pids": pids,
        "qids": qids,
        "doc_texts": doc_texts,
        "q_texts": q_texts,
        "qid_to_idx": {q: i for i, q in enumerate(qids)},
        "pid_to_text": dict(zip(pids, doc_texts)),
        "qid_to_text": dict(zip(qids, q_texts)),
        "qrels": qrels,
        "example_qids": example_qids,
    }
    print(f"{name}: docs={len(pids)} queries={len(qids)} example_qids={len(example_qids)}")


[beir] nfcorpus уже готов: /home/uvuv/workspace/sandbox_03/data/beir/nfcorpus
[beir] scifact уже готов: /home/uvuv/workspace/sandbox_03/data/beir/scifact
[progress] лог прогресса: /home/uvuv/workspace/sandbox_03/outputs/logs/progress_20260615-143736.log
nfcorpus: docs=3633 queries=323 example_qids=10
scifact: docs=5183 queries=300 example_qids=10


## 6. Eval-хелперы и формат логов

Каждый checkpoint → один JSON `step_XXXXXX.json`:
- `step`, `train_loss`
- `nfcorpus` / `scifact`: aggregate (ndcg@10, mrr@10, recall@10/100, sparsity) + 10 примеров с ranking


In [6]:
def gpu_cleanup():
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()


def _per_query_lookup(pq, qid):
    try:
        i = pq["qids"].index(qid)
    except ValueError:
        return {}
    return {k: float(pq[k][i]) for k in pq if k != "qids"}


def _build_examples(ranks, cache, example_qids, topk=RANK_TOPK):
    examples = []
    for qid in example_qids:
        i = cache["qid_to_idx"][qid]
        rel_map = cache["qrels"].get(qid, {})
        rel_set = set(rel_map)
        ranked_pids = [cache["pids"][j] for j in ranks[i]]
        ranking = []
        for rank, pid in enumerate(ranked_pids[:topk], start=1):
            ranking.append({
                "rank": rank,
                "pid": pid,
                "relevant": pid in rel_set,
                "rel_grade": rel_map.get(pid, 0),
                "doc_preview": cache["pid_to_text"][pid][:200],
            })
        examples.append({
            "qid": qid,
            "query": cache["qid_to_text"][qid][:200],
            "n_relevant": len(rel_set),
            "ranking": ranking,
        })
    return examples


def eval_beir_dataset(model, tokenizer, cfg, dataset_name):
    cache = BEIR_CACHE[dataset_name]
    ecfg, mcfg = cfg["eval"], cfg["model"]
    was_training = model.training
    model.eval()

    d_mat = evaluate.encode_sparse(
        model, tokenizer, cache["doc_texts"], mcfg["max_len_doc"],
        ecfg["batch_size_docs"], device, kind="doc")
    q_mat = evaluate.encode_sparse(
        model, tokenizer, cache["q_texts"], mcfg["max_len_query"],
        ecfg["batch_size_queries"], device, kind="query")

    topk = max(max(RECALL_KS), max(NDCG_KS), RANK_TOPK)
    ranks = evaluate.search(q_mat, d_mat, topk, device, ecfg["batch_size_search"])

    pq = benchmark.per_query_metrics(
        ranks, cache["qids"], cache["pids"], cache["qrels"],
        ks=RECALL_KS, ndcg_ks=NDCG_KS)
    agg = benchmark.aggregate(pq)
    agg["n_corpus_docs"] = len(cache["pids"])
    agg["avg_nnz_doc"] = float(d_mat.getnnz(axis=1).mean())
    agg["avg_nnz_query"] = float(q_mat.getnnz(axis=1).mean())

    examples = _build_examples(ranks, cache, cache["example_qids"])
    for ex in examples:
        ex["metrics"] = _per_query_lookup(pq, ex["qid"])

    result = {"aggregate": agg, "examples": examples}

    del d_mat, q_mat, ranks, pq
    gpu_cleanup()
    if was_training:
        model.train()
    return result


def save_step_log(log_dir, step, train_loss, beir_results):
    payload = {
        "step": step,
        "train_loss": train_loss,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "metrics_schema": {
            "aggregate": ["ndcg@10", "mrr@10", "recall@10", "recall@100",
                          "avg_nnz_doc", "avg_nnz_query", "n_eval_queries", "n_corpus_docs"],
            "per_query": ["ndcg@10", "mrr@10", "recall@10", "recall@100"],
        },
    }
    for name in BEIR_NAMES:
        payload[name] = beir_results[name]
    path = log_dir / f"step_{step:06d}.json"
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    return path


def run_beir_eval(model, tokenizer, cfg, step, train_loss, log_dir):
    beir_results = {}
    for name in BEIR_NAMES:
        beir_results[name] = eval_beir_dataset(model, tokenizer, cfg, name)
    path = save_step_log(log_dir, step, train_loss, beir_results)
    nfc = beir_results["nfcorpus"]["aggregate"]
    sci = beir_results["scifact"]["aggregate"]
    print(f"[eval@{step}] nfc ndcg@10={nfc['ndcg@10']:.4f} mrr@10={nfc['mrr@10']:.4f} | "
          f"sci ndcg@10={sci['ndcg@10']:.4f} mrr@10={sci['mrr@10']:.4f} -> {path.name}")
    return beir_results


## 7. Обучение + периодический BEIR-eval

Модель остаётся in-memory; промежуточные веса на диск не пишутся. После каждого eval
— явная очистка GPU/RAM.


In [7]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
log_dir = OUTPUTS_DIR / "logs" / run_id
log_dir.mkdir(parents=True, exist_ok=True)
(log_dir / "config.json").write_text(json.dumps(cfg, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"[run] {cfg['version']} mode={MODE} device={device}")
print(f"[run] логи BEIR-eval -> {log_dir}")

set_seed(cfg["train"]["seed"])
tokenizer = AutoTokenizer.from_pretrained(cfg["model"]["hf_model"])
model = Splade(cfg["model"]["hf_model"], cfg["model"]["query_encoder"]).to(device)

tcfg, mcfg = cfg["train"], cfg["model"]
triples = data.load_triples(msmarco_dir)
rng = np.random.default_rng(tcfg["seed"])
order = rng.permutation(len(triples))
optimizer = torch.optim.AdamW(model.parameters(), lr=tcfg["lr"])
max_steps, warmup = tcfg["max_steps"], tcfg["warmup_steps"]

def lr_lambda(step):
    if step < warmup:
        return (step + 1) / max(1, warmup)
    return max(0.0, (max_steps - step) / max(1, max_steps - warmup))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
bs = tcfg["batch_size"]
use_amp = device.type == "cuda"
model.train()
losses, ptr = [], 0

pbar = tqdm(range(max_steps), desc=f"train:{cfg['version']}", unit=" шаг")
for step in pbar:
    if ptr + bs > len(order):
        order = rng.permutation(len(triples))
        ptr = 0
    batch = [triples[i] for i in order[ptr:ptr + bs]]
    ptr += bs
    queries = [t[0] for t in batch]
    docs = [t[1] for t in batch] + [t[2] for t in batch]

    q_enc = tok(tokenizer, queries, mcfg["max_len_query"], device, special=True)
    d_enc = tok(tokenizer, docs, mcfg["max_len_doc"], device)
    lam_scale = min(1.0, ((step + 1) / max(1, tcfg["flops_warmup_steps"])) ** 2)

    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=use_amp):
        q = model.encode_queries(q_enc["input_ids"], q_enc["attention_mask"],
                                 q_enc.get("special_tokens_mask"))
        d = model.encode_docs(d_enc["input_ids"], d_enc["attention_mask"])
        scores = q @ d.t()
        labels = torch.arange(len(batch), device=device)
        loss = F.cross_entropy(scores, labels)
        if model.query_encoder == "mlm" and tcfg["lambda_q"] > 0:
            loss = loss + lam_scale * tcfg["lambda_q"] * flops_loss(q)
        if tcfg["lambda_d"] > 0:
            loss = loss + lam_scale * tcfg["lambda_d"] * flops_loss(d)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    losses.append(loss.item())

    if step == 0 or (step + 1) % tcfg["log_every"] == 0:
        window = losses[-tcfg["log_every"]:]
        pbar.set_postfix(loss=f"{np.mean(window):.4f}",
                         lr=f"{scheduler.get_last_lr()[0]:.2e}")

    if (step + 1) % EVAL_EVERY == 0:
        recent = losses[-min(len(losses), tcfg["log_every"]):]
        run_beir_eval(model, tokenizer, cfg, step + 1, float(np.mean(recent)), log_dir)

final_loss = float(np.mean(losses[-10:]))
summary = {
    "run_id": run_id,
    "version": cfg["version"],
    "mode": MODE,
    "max_steps": max_steps,
    "eval_every": EVAL_EVERY,
    "final_train_loss": final_loss,
    "log_dir": str(log_dir),
    "n_log_files": len(list(log_dir.glob("step_*.json"))),
}
(log_dir / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\n[run] готово | final_loss={final_loss:.4f} | логов: {summary['n_log_files']} | {log_dir}")


[run] v1_splade_max mode=full device=cuda
[run] логи BEIR-eval -> /home/uvuv/workspace/sandbox_03/outputs/logs/20260614-211756


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /distilbert-base-uncased/resolve/main/tokenizer_config.json (Caused by NewConnectionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to establish a new connection: [Errno 13] Permission denied"))'), '(Request ID: 15881b04-2179-42a8-a24c-e6a8f85fd0c5)')' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json


Retrying in 1s [Retry 1/5].


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /distilbert-base-uncased/resolve/main/tokenizer_config.json (Caused by NewConnectionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to establish a new connection: [Errno 13] Permission denied"))'), '(Request ID: b77f9e65-ee7a-46c5-bf43-fa04bd7a8f65)')' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json


Retrying in 2s [Retry 2/5].


...

[eval@396000] nfc ndcg@10=0.3090 mrr@10=0.5015 | sci ndcg@10=0.5971 mrr@10=0.5611 -> step_396000.json


[eval@397000] nfc ndcg@10=0.3084 mrr@10=0.4997 | sci ndcg@10=0.5981 mrr@10=0.5614 -> step_397000.json


[eval@398000] nfc ndcg@10=0.3077 mrr@10=0.5006 | sci ndcg@10=0.5980 mrr@10=0.5617 -> step_398000.json


[eval@399000] nfc ndcg@10=0.3082 mrr@10=0.5010 | sci ndcg@10=0.5981 mrr@10=0.5618 -> step_399000.json


[eval@400000] nfc ndcg@10=0.3076 mrr@10=0.4989 | sci ndcg@10=0.5980 mrr@10=0.5617 -> step_400000.json

[run] готово | final_loss=0.0015 | логов: 400 | /home/uvuv/workspace/sandbox_03/outputs/logs/20260614-211756


## 8. Сводка по логам BEIR-eval


In [10]:
rows = []

log_dir = Path('/home/uvuv/workspace/sandbox_03/outputs/logs/20260614-211756/')

for path in sorted(log_dir.glob("step_*.json")):
    rec = json.loads(path.read_text(encoding="utf-8"))
    row = {"step": rec["step"], "train_loss": rec.get("train_loss")}
    for name in BEIR_NAMES:
        agg = rec[name]["aggregate"]
        for k, v in agg.items():
            if isinstance(v, float):
                row[f"{name}/{k}"] = v
    rows.append(row)

df = pd.DataFrame(rows).sort_values("step").reset_index(drop=True)
out_csv = log_dir / "metrics_overview.csv"
df.to_csv(out_csv, index=False)
print(df.to_string(index=False))
print(f"\n[logs] CSV: {out_csv}")
df


  step  train_loss  nfcorpus/mrr@10  nfcorpus/recall@10  nfcorpus/recall@100  nfcorpus/ndcg@10  nfcorpus/avg_nnz_doc  nfcorpus/avg_nnz_query  scifact/mrr@10  scifact/recall@10  scifact/recall@100  scifact/ndcg@10  scifact/avg_nnz_doc  scifact/avg_nnz_query
  1000    0.993505         0.442435            0.123703             0.232102          0.252473            961.897881              163.990712        0.423165           0.583056            0.769944         0.456320           879.694771             446.500000
  2000    0.228975         0.504407            0.147738             0.263142          0.304917            489.342692               50.130031        0.516884           0.690444            0.833667         0.555226           478.628401             203.526667
  3000    0.152077         0.514029            0.150436             0.273336          0.312579            485.825764               37.950464        0.547533           0.692111            0.855333         0.577822           476.64

,step,train_loss,nfcorpus/mrr@10,nfcorpus/recall@10,nfcorpus/recall@100,nfcorpus/ndcg@10,nfcorpus/avg_nnz_doc,nfcorpus/avg_nnz_query,scifact/mrr@10,scifact/recall@10,scifact/recall@100,scifact/ndcg@10,scifact/avg_nnz_doc,scifact/avg_nnz_query
0,1000,0.993505,0.442435,0.123703,0.232102,0.252473,961.897881,163.990712,0.423165,0.583056,0.769944,0.456320,879.694771,446.500000
1,2000,0.228975,0.504407,0.147738,0.263142,0.304917,489.342692,50.130031,0.516884,0.690444,0.833667,0.555226,478.628401,203.526667
2,3000,0.152077,0.514029,0.150436,0.273336,0.312579,485.825764,37.950464,0.547533,0.692111,0.855333,0.577822,476.644029,175.283333
3,4000,0.125357,0.523576,0.150455,0.271877,0.317901,605.244701,32.325077,0.565439,0.694000,0.862000,0.587904,598.621262,165.786667
4,5000,0.102299,0.512077,0.155967,0.269375,0.321470,839.379851,36.931889,0.566394,0.690944,0.850333,0.588752,795.361374,173.193333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,396000,0.006650,0.501477,0.153113,0.257937,0.308961,197.682631,24.241486,0.561114,0.726667,0.903667,0.597141,213.430446,79.843333
396,397000,0.008000,0.499722,0.153041,0.257942,0.308408,196.649050,24.117647,0.561415,0.730000,0.903667,0.598111,212.400733,79.410000
397,398000,0.005978,0.500634,0.152499,0.257916,0.307715,197.489403,24.061920,0.561702,0.728333,0.903667,0.598007,213.206637,79.590000
398,399000,0.008440,0.500961,0.152654,0.257889,0.308166,197.792733,24.055728,0.561804,0.728333,0.903667,0.598073,213.306386,79.576667
